In [ ]:
!pip install roboflow

from roboflow import Roboflow

# Initialize roboflow with your API key
rf = Roboflow(api_key="56ROJMsyNuusXttM8RAf")
project = rf.workspace("coursework-y7x7o").project("eye-disease-classification-iu3lf")
version = project.version(2)

# Download the dataset in folder format (this creates proper train/valid/test folders with class subfolders)
dataset = version.download("folder")

print(f"Dataset downloaded to: {dataset.location}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 108.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Eye-disease-classification-2 in folder:: 100%|██████████| 1196/1196 [00:00<00:00, 5422.59it/s]

Dataset downloaded to: /content/Eye-disease-classification-2


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
import numpy as np
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# TRANSFORMS (reduced for medical images)

transform_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(6),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

transform_eval = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])


# LOAD DATA

data_path = "/content/Eye-disease-classification-2"

full_train = datasets.ImageFolder(os.path.join(data_path, "train"), transform=transform_train)
test_dataset = datasets.ImageFolder(os.path.join(data_path, "test"), transform=transform_eval)

val_size = int(0.2 * len(full_train))
train_size = len(full_train) - val_size
train_dataset, valid_dataset = random_split(full_train, [train_size, val_size])
valid_dataset.dataset.transform = transform_eval

print("Classes:", full_train.classes)


# OVERSAMPLING FOR TRAIN SUBSET

train_indices = train_dataset.indices
targets = [full_train[idx][1] for idx in train_indices]

class_counts = np.bincount(targets)
print("Train subset class counts:", class_counts)

class_weights = 1.0 / class_counts
sample_weights = [class_weights[t] for t in targets]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
valid_loader = DataLoader(valid_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)


# MODEL

model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features

model.fc = nn.Sequential(
    nn.Dropout(0.4),            # more regularization
    nn.Linear(num_ftrs, len(full_train.classes))
)
model = model.to(device)


# STAGE 1 – TRAIN ONLY FC LAYER

criterion_stage1 = nn.CrossEntropyLoss(label_smoothing=0.1)

for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.fc.parameters(), lr=9e-4, weight_decay=2e-4)

print("\n🔹 Stage 1 Training...")
for epoch in range(3):
    model.train()
    running_loss, correct, total = 0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion_stage1(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, pred = out.max(1)
        correct += pred.eq(y).sum().item()
        total += y.size(0)

    # Validation
    val_correct, val_total = 0, 0
    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            _, pred = out.max(1)
            val_correct += pred.eq(y).sum().item()
            val_total += y.size(0)

    print(f"[Stage1] Epoch {epoch+1}/3 "
          f"Loss: {running_loss/len(train_loader):.4f}  "
          f"Train Acc: {100*correct/total:.2f}%  "
          f"Val Acc: {100*val_correct/val_total:.2f}%")


# STAGE 2 – PARTIAL FINE-TUNING

criterion = nn.CrossEntropyLoss()

for name, param in model.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=6e-6,             # even smaller LR → less overfitting
    weight_decay=5e-4
)

best_val = 0
patience = 3
count = 0

print("\n🔹 Stage 2 Fine-tuning...")
for epoch in range(10):
    model.train()
    running_loss, correct, total = 0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, pred = out.max(1)
        correct += pred.eq(y).sum().item()
        total += y.size(0)

    # Validation
    val_correct, val_total = 0, 0
    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            _, pred = out.max(1)
            val_correct += pred.eq(y).sum().item()
            val_total += y.size(0)

    val_acc = 100 * val_correct / val_total
    print(f"[Stage2] Epoch {epoch+1}/10  "
          f"Loss: {running_loss/len(train_loader):.4f}  "
          f"Train Acc: {100*correct/total:.2f}%  "
          f"Val Acc: {val_acc:.2f}%")

    if val_acc > best_val + 0.2:
        best_val = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        count = 0
    else:
        count += 1
        if count >= patience:
            print("Early stopping triggered!")
            break


# FINAL TEST ACCURACY ONLY

model.load_state_dict(torch.load("best_model.pth"))
model.eval()

test_correct, test_total = 0, 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        _, pred = out.max(1)
        test_correct += pred.eq(y).sum().item()
        test_total += y.size(0)

print(f"\n FINAL TEST ACCURACY: {100*test_correct/test_total:.2f}%")

Using device: cpu
Classes: ['Cataracts', 'Normal_Eyes', 'Uveitis']
Train subset class counts: [229 290 251]
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 146MB/s]



🔹 Stage 1 Training...
[Stage1] Epoch 1/3 Loss: 1.1385  Train Acc: 42.99%  Val Acc: 61.98%
[Stage1] Epoch 2/3 Loss: 0.9967  Train Acc: 54.81%  Val Acc: 64.06%
[Stage1] Epoch 3/3 Loss: 0.9143  Train Acc: 62.08%  Val Acc: 74.48%

🔹 Stage 2 Fine-tuning...
[Stage2] Epoch 1/10  Loss: 0.7582  Train Acc: 66.49%  Val Acc: 75.00%
[Stage2] Epoch 2/10  Loss: 0.7506  Train Acc: 70.52%  Val Acc: 77.60%
[Stage2] Epoch 3/10  Loss: 0.6461  Train Acc: 77.14%  Val Acc: 76.56%
[Stage2] Epoch 4/10  Loss: 0.5943  Train Acc: 80.52%  Val Acc: 79.69%
[Stage2] Epoch 5/10  Loss: 0.5429  Train Acc: 82.73%  Val Acc: 78.65%
[Stage2] Epoch 6/10  Loss: 0.5139  Train Acc: 83.12%  Val Acc: 80.21%
[Stage2] Epoch 7/10  Loss: 0.4931  Train Acc: 82.60%  Val Acc: 79.17%
[Stage2] Epoch 8/10  Loss: 0.4476  Train Acc: 83.64%  Val Acc: 81.77%
[Stage2] Epoch 9/10  Loss: 0.4731  Train Acc: 86.23%  Val Acc: 80.21%
[Stage2] Epoch 10/10  Loss: 0.3819  Train Acc: 88.44%  Val Acc: 81.25%

 FINAL TEST ACCURACY: 85.91%
